# ArchiGen: SD1.5 facade ControlNet, Kaggle → MLflow → local PC

This experiment fine-tunes a **ControlNet** while freezing SD1.5, VAE and CLIP. It is a candidate, not a proven improvement. CMP has no labels for our four named styles; text styling uses the pretrained prior.

**Before Run All:** attach the private `khalilayeb/archigen-cmp-facades-original-locked-splits` dataset; enable Internet and a GPU. One GPU is used even on a dual-T4 machine. The 400/100/106 splits are locked. Test data is not used in training or checkpoint selection.

The source is checked out from the GitHub experiment branch at a recorded commit. MLflow defaults to a Kaggle-local SQLite store with portable events for import into your existing PC database. Optional remote tracking credentials belong in Kaggle Secrets, never this notebook.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path
REPO = "https://github.com/ayebkalil/archigen-3d.git"
BRANCH = "codex/facade-controlnet-kaggle-mlflow"
PINNED_COMMIT = ""
REPO_DIR = Path("/kaggle/working/archigen-source")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO, str(REPO_DIR)], check=True)
if PINNED_COMMIT:
    subprocess.run(["git", "fetch", "--depth", "1", "origin", PINNED_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO_DIR, check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Source branch:", BRANCH, "commit:", SOURCE_COMMIT)
EXPERIMENT = REPO_DIR / "experiments/facade_sd15_controlnet"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(EXPERIMENT / "requirements-train.txt")], check=True)


## Optional secrets and GPU check
No remote URI is needed for the default portable workflow. For live remote MLflow, add `MLFLOW_TRACKING_URI` and the required tracking username/password or token to Kaggle Secrets. A PC's `localhost:5000` is not reachable from Kaggle. `HF_TOKEN` is optional for these public pinned model repositories.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD", "MLFLOW_TRACKING_TOKEN", "HF_TOKEN"):
        try:
            value = secrets.get_secret(key)
            if value: os.environ[key] = value
        except Exception:
            pass
except ImportError:
    pass
from urllib.parse import urlsplit
if os.getenv("MLFLOW_TRACKING_URI") and urlsplit(os.environ["MLFLOW_TRACKING_URI"]).hostname in {"localhost", "127.0.0.1"}:
    raise ValueError("A PC localhost MLflow URI cannot be used from Kaggle. Remove the secret for portable local tracking.")
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU before training"
print("Torch:", torch.__version__, "GPU 0:", torch.cuda.get_device_name(0))
print("Visible GPUs:", torch.cuda.device_count(), "; training uses one GPU, with batch 1 and accumulation 4.")
print("GPU 0 VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)
print("Tracking:", "remote (URI hidden)" if os.getenv("MLFLOW_TRACKING_URI") else "Kaggle SQLite plus portable MLflow events")


## Locate and verify the original-resolution dataset
The training module verifies photo/mask checksums and duplicate identities. Original photographs are retained; some source images are smaller than 512 on one axis. Resizing does not create new ground-truth detail.

In [ ]:
sys.path.insert(0, str(REPO_DIR))
from experiments.facade_sd15_controlnet.data import locate_kaggle_dataset
DATA_ROOT = locate_kaggle_dataset()
subprocess.run([sys.executable, "-m", "experiments.facade_sd15_controlnet.manage", "check-data", "--data", str(DATA_ROOT)], cwd=REPO_DIR, check=True)


## Train or smoke-test
`MODE="train"` runs the configured 1500 optimizer steps; runtime depends on the assigned GPU. Use `MODE="smoke"` for two actual optimizer steps first. Smoke checks do not export a deployable model and do not establish quality.

Checkpoint every 250 steps, keep two resumable checkpoints, save the lowest fixed-subset validation LPIPS candidate. To resume an interrupted run, attach its output and copy its complete run folder (including best-controlnet and checkpoints) to RUN_DIR before setting RESUME. Keep the same config/dataset. A resumed segment gets an MLflow run linked to its parent run.

In [ ]:
MODE = "train"  # choose "smoke" for the first short GPU check
RUN_DIR = Path("/kaggle/working/facade-controlnet")
RESUME = None  # e.g. RUN_DIR / "checkpoints/step_000750", after copying the previous full run output
command = [sys.executable, "-m", "experiments.facade_sd15_controlnet.train", "--data", str(DATA_ROOT), "--output", str(RUN_DIR), "--config", str(EXPERIMENT / "config.json")]
if MODE == "smoke": command += ["--smoke"]
if RESUME: command += ["--resume", str(RESUME)]
subprocess.run(command, cwd=REPO_DIR, check=True)


## Evaluate the selected candidate on validation, then download
This evaluates all 100 validation images with fixed seeds. If the dataset package includes the current production ONNX baseline, paired baseline metrics and comparisons are produced. Training/selection used a small validation subset, so validation is not an independent test. The 106-image test split remains untouched; run it only after model selection using the documented `--split test --final-test` command.

Reported opening-edge MAE is a proxy, not segmentation IoU. Photorealism, opening counts and local speed still require inspection.

In [ ]:
if MODE == "train":
    subprocess.run([sys.executable, "-m", "experiments.facade_sd15_controlnet.evaluate", "--model", str(RUN_DIR / "export"), "--data", str(DATA_ROOT), "--output", str(RUN_DIR / "evaluation")], cwd=REPO_DIR, check=True)
    print((RUN_DIR / "evaluation/evaluation.json").read_text())
else:
    print("Smoke complete: inspect training loss/MLflow events; no quality claim or deployable model.")


In [ ]:
if MODE == "train":
    ARCHIVE = Path("/kaggle/working/archigen-facade-model-and-mlflow.zip")
    subprocess.run([sys.executable, "-m", "experiments.facade_sd15_controlnet.manage", "bundle", "--run", str(RUN_DIR), "--archive", str(ARCHIVE)], cwd=REPO_DIR, check=True)
    from IPython.display import FileLink, display
    display(FileLink(str(ARCHIVE)))
    print("Download the archive for local install. Keep the run folder/checkpoints in the saved Kaggle output for resuming.")


## Run on your RTX 3050 laptop
From this experiment branch, install the downloaded archive with `python -m experiments.facade_sd15_controlnet.manage install --archive PATH.zip --destination models/saved/facade-controlnet-kaggle --tracking-uri sqlite:///mlflow.db`.

Use a separate diffusion Python environment, set `ARCHIGEN_CONTROLNET_MODEL` to the installed **export** folder, and start the worker on `127.0.0.1:8002`. Set `ARCHIGEN_FACADE_CONTROLNET_URL=http://127.0.0.1:8002` only in the backend you want to test. The README contains complete Windows commands.

The first local use also needs the pinned frozen SD1.5 base weights (download once or transfer a cache). CPU offload, FP16 and tiled VAE target 4GB VRAM; profile the downloaded candidate before claiming it meets the laptop budget. The current project remains on Pix2Pix unless explicitly enabled.